# Klasifikasi hierarkis: label bertingkat (kategori > subkategori)
**Kapan dipakai:** label punya pohon, mis. kategori produk "Tops > Knits", topik berita
"Ekonomi > Saham", keluhan "Layanan > Tagihan". Kolom tingkat atas ada di train (atau satu kolom
jalur "a > b > c"); metrik lomba biasanya di tingkat paling bawah (daun).

Lima cara dibandingkan di lipatan yang sama (TF-IDF kata + huruf, SGD log-loss):
1. **datar**: langsung tebak daun, pohon diabaikan.
2. **konsisten per tingkat**: satu model per tingkat, lalu pilih jalur sah dengan hasil kali
   peluang tiap tingkat (tidak pernah menebak "Tops > Jeans").
3. **top-down keras**: tebak induk dulu, lalu daun hanya di antara anak induk itu.
4. **top-down lunak (rantai)**: P(daun) = P(induk) × P(daun | induk), model lokal per induk.
5. **gabung datar + rantai**: rata-rata peluang cara 1 dan 4.

Metrik: skor daun (metrik lomba), F1 tingkat atas, F1 hierarkis (porsi leluhur yang benar), dan
porsi "salah tapi saudara" (salah daun, induk benar). **Data demo:** ulasan baju, tebak
`class_name` (20) di bawah `department_name` (6) dari teks saja. Kolom tabel lain: pakai OOF
notebook ini sebagai `EXTRA_OOF` di `stacking_teks_tabular`.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack, vstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/klasifikasi_hierarkis"
OOF_FOLDER = "../../../outputs/text/oof"

DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample dari panitia
KEY_CSV = TEST_CSV                # demo: test baju memuat class_name (GANTI saat lomba / hapus)

ID_COL = "id"                                                # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]                         # GANTI saat lomba
LEVELS = ["department_name", "class_name"]                   # GANTI saat lomba: atas → bawah
# LEVELS = ["division_name", "department_name", "class_name"]   # alt: 3 tingkat
TARGET = LEVELS[-1]               # kolom yang dinilai (daun), boleh kolom jalur asli
SUB_COL = TARGET                  # GANTI saat lomba: kolom sample
PATH_SEP = " > "

MODEL_NAME = "tfidf_hierarki"
N_FOLDS = 5
SEED = 42
MAX_PLOT_LEAVES = 40              # matriks bingung digambar bila daun <= ini
BASE_MODEL = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)  # 5 dtk/model
# BASE_MODEL = LogisticRegression(C=4.0, solver="saga", max_iter=200, tol=1e-3)  # 30 dtk/model
# BASE_MODEL = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced")  # kelas kecil

In [ ]:
def metric(truth, labels):
    return f1_score(truth, labels, average="macro")       # GANTI: metrik lomba
    # return accuracy_score(truth, labels)


print("uji metrik:", metric([0, 1, 1], [0, 1, 0]))

## Baca data
Label berupa satu kolom jalur ("a > b > c")? Buka baris alt: kolom dipecah per tingkat, lalu
`LEVELS = ["tingkat_0", "tingkat_1", ...]` dan `TARGET` tetap kolom jalur asli.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
# levels_split = train["kategori"].str.split(PATH_SEP, expand=True)     # alt: kolom jalur
# for number in range(levels_split.shape[1]):
#     train[f"tingkat_{number}"] = levels_split[number]
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in LEVELS + [TARGET]:
    train[col] = train[col].fillna("kosong").astype(str)
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
N_LEVELS = len(LEVELS)
print("train:", train.shape, "| test:", test.shape, "| tingkat:", LEVELS)

## Bangun pohon
Simpul = jalur penuh ("Tops > Knits"), jadi nama anak yang sama di dua induk tetap dua simpul
dan pohon selalu sah. `parent_of[tingkat][anak] = induk`, `ancestor[tingkat][daun] = leluhur`.

In [ ]:
level_path = []
current = train[LEVELS[0]]
for level, col in enumerate(LEVELS):
    if level > 0:
        current = current + PATH_SEP + train[col]
    level_path.append(current)
NODES = []
node_id = []
n_nodes = []
parent_of = [None]
for level in range(N_LEVELS):
    NODES.append(np.unique(level_path[level]))
    node_id.append(np.searchsorted(NODES[level], level_path[level]))
    n_nodes.append(len(NODES[level]))
for level in range(1, N_LEVELS):
    links = np.zeros(n_nodes[level], dtype=int)
    links[node_id[level]] = node_id[level - 1]
    parent_of.append(links)
LEAVES = NODES[-1]
ancestor = [None] * N_LEVELS
ancestor[-1] = np.arange(n_nodes[-1])
for level in range(N_LEVELS - 1, 0, -1):
    ancestor[level - 1] = parent_of[level][ancestor[level]]
print("simpul per tingkat:", n_nodes)

Peta daun → kelas `TARGET` (format OOF bersama: kolom = `np.unique(train[TARGET])`) dan cek
nama anak yang punya lebih dari satu induk.

In [ ]:
CLASSES = np.unique(train[TARGET])
y_idx = np.searchsorted(CLASSES, train[TARGET])
leaf_class = np.zeros(n_nodes[-1], dtype=int)
leaf_class[node_id[-1]] = y_idx
to_class = np.eye(len(CLASSES), dtype="float32")[leaf_class]      # daun × kelas
leaf_short = np.empty(n_nodes[-1], dtype=object)
leaf_short[node_id[-1]] = train[LEVELS[-1]].values
for level in range(1, N_LEVELS):
    parents = train.groupby(LEVELS[level])[LEVELS[level - 1]].nunique()
    shared = parents[parents > 1]
    print(f"{LEVELS[level]}: {len(shared)} nama punya > 1 induk", list(shared.index[:5]))
counts = pd.Series(np.bincount(node_id[-1]), index=LEAVES, name="baris")
counts.to_csv(f"{OUTPUT_FOLDER}/hirarki.csv")
smallest = counts.sort_values()
print("daun terkecil:", smallest.head(5).to_dict())

## Gambar pohon (bahan slide): cincin dalam = tingkat atas, cincin luar = daun

In [ ]:
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#999999", "#F0E442"]
top_colors = []
for number in range(n_nodes[0]):
    top_colors.append(PALETTE[number % len(PALETTE)])
top_counts = np.bincount(node_id[0], minlength=n_nodes[0])
top_text = np.where(top_counts / top_counts.sum() >= 0.02, NODES[0], "")
leaf_counts = counts.to_numpy()
leaf_text = []
for number in range(n_nodes[-1]):
    label = ""
    if leaf_counts[number] / leaf_counts.sum() >= 0.025:
        label = leaf_short[number]
    leaf_text.append(label)
figure, ax = plt.subplots(figsize=(8, 8))
ax.pie(top_counts, radius=0.7, colors=top_colors, labels=top_text, labeldistance=0.75,
       wedgeprops={"width": 0.3, "edgecolor": "white"}, textprops={"fontsize": 9})
ax.pie(leaf_counts, radius=1.0, colors=np.array(top_colors)[ancestor[0]], labels=leaf_text,
       wedgeprops={"width": 0.3, "edgecolor": "white", "alpha": 0.7}, textprops={"fontsize": 8})
ax.set_title(f"{' > '.join(LEVELS)}: {len(train)} baris, {n_nodes[-1]} daun")
figure.savefig(f"{OUTPUT_FOLDER}/hirarki.png", dpi=200, bbox_inches="tight")
plt.show()

## Lipatan + TF-IDF

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
             .split(np.zeros(len(y_idx)), y_idx))
all_text = pd.concat([full_train, full_test], ignore_index=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=100000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
print("TF-IDF:", matrix_train.shape)

## Model per tingkat (datar) + model lokal per induk
`fit_proba` menaruh peluang di kolom simpul global; kelas yang tidak ada di lipatan = 0.
`local_block`: untuk tiap induk, model dilatih hanya pada baris induk itu dan menebak anaknya,
hasilnya P(anak | induk) untuk **semua** baris (dipakai rantai dan top-down).

In [ ]:
def fit_proba(rows, labels, query, n_columns):
    out = np.zeros((query.shape[0], n_columns), dtype="float32")
    present = np.unique(labels)
    if len(present) == 1:
        out[:, present[0]] = 1.0
        return out
    fitted = clone(BASE_MODEL).fit(matrix_train[rows], labels)
    out[:, fitted.classes_] = fitted.predict_proba(query)
    return out


def local_block(level, idx_fit, query):
    out = np.zeros((query.shape[0], n_nodes[level]), dtype="float32")
    fit_parent = node_id[level - 1][idx_fit]
    for parent in range(n_nodes[level - 1]):
        children = np.flatnonzero(parent_of[level] == parent)
        rows = idx_fit[fit_parent == parent]
        if len(rows) == 0:
            out[:, children] = 1.0 / len(children)
            continue
        out += fit_proba(rows, node_id[level][rows], query, n_nodes[level])
    return out


check = local_block(N_LEVELS - 1, folds[0][0][:3000], matrix_train[:3])
print("uji: jumlah per baris =", check.sum(axis=1).round(3), "| harus = jumlah induk",
      n_nodes[-2])

Jalankan di lima lipatan. Tingkat paling atas: model lokal = model datar (induknya akar).

In [ ]:
oof_flat, test_flat, oof_cond, test_cond = [], [], [], []
for level in range(N_LEVELS):
    oof_flat.append(np.zeros((len(train), n_nodes[level]), dtype="float32"))
    test_flat.append(np.zeros((len(test), n_nodes[level]), dtype="float32"))
    oof_cond.append(np.zeros((len(train), n_nodes[level]), dtype="float32"))
    test_cond.append(np.zeros((len(test), n_nodes[level]), dtype="float32"))
START = time.time()
for fold, (idx_fit, idx_val) in enumerate(folds):
    query = vstack([matrix_train[idx_val], matrix_test]).tocsr()
    n_val = len(idx_val)
    for level in range(N_LEVELS):
        proba = fit_proba(idx_fit, node_id[level][idx_fit], query, n_nodes[level])
        oof_flat[level][idx_val] = proba[:n_val]
        test_flat[level] += proba[n_val:] / N_FOLDS
        if level > 0:
            proba = local_block(level, idx_fit, query)
        oof_cond[level][idx_val] = proba[:n_val]
        test_cond[level] += proba[n_val:] / N_FOLDS
    print("lipatan", fold, "| detik", round(time.time() - START))
SECONDS = round(time.time() - START)

## Lima cara menggabungkan
Semua menghasilkan peluang per daun, jadi dinilai dengan kode yang sama.

In [ ]:
def path_product(per_level):
    product = np.ones((per_level[0].shape[0], n_nodes[-1]), dtype="float64")
    for level in range(N_LEVELS):
        product *= per_level[level][:, ancestor[level]]
    return product / np.clip(product.sum(axis=1, keepdims=True), 1e-12, None)


def top_down_hard(cond):
    chosen = cond[0].argmax(axis=1)
    for level in range(1, N_LEVELS):
        allowed = parent_of[level][None, :] == chosen[:, None]
        chosen = np.where(allowed, cond[level], -1.0).argmax(axis=1)
    return np.eye(n_nodes[-1], dtype="float32")[chosen]


def all_methods(flat, cond):
    chain = path_product(cond)
    return {"datar": flat[-1], "konsisten per tingkat": path_product(flat),
            "top-down keras": top_down_hard(cond), "top-down lunak (rantai)": chain,
            "gabung datar + rantai": (flat[-1] + chain) / 2}


oof_methods = all_methods(oof_flat, oof_cond)
test_methods = all_methods(test_flat, test_cond)
print(list(oof_methods))

## Evaluasi OOF
F1 hierarkis = rata-rata porsi simpul jalur tebakan yang juga ada di jalur asli (semua jalur sama
dalam, jadi presisi = recall). "Salah tapi saudara" = di antara daun yang salah, porsi yang
induknya benar: tinggi berarti model paham kategori besar, hanya bingung di dalamnya.
Skor daun memakai peluang yang dijumlah per kelas `TARGET` (sama dengan OOF + submission).
Pakai hanya tingkat yang sungguh induk: di baju, `division_name` (ukuran, bukan kategori)
sebagai tingkat 3 membuat top-down keras turun (0,411 vs datar 0,426) dan 38% jalur tak sah.

In [ ]:
def evaluate(leaf_proba, true_leaf, truth_class):
    pred_leaf = leaf_proba.argmax(axis=1)
    shared = np.zeros(len(true_leaf))
    for level in range(N_LEVELS):
        shared += ancestor[level][pred_leaf] == ancestor[level][true_leaf]
    wrong = pred_leaf != true_leaf
    sibling = ancestor[-2][pred_leaf] == ancestor[-2][true_leaf]
    return {"skor daun (metrik)": metric(truth_class, (leaf_proba @ to_class).argmax(axis=1)),
            "akurasi jalur": np.mean(~wrong),
            "F1 tingkat atas": f1_score(ancestor[0][true_leaf], ancestor[0][pred_leaf],
                                        average="macro"),
            "F1 hierarkis": shared.mean() / N_LEVELS,
            "salah tapi saudara": sibling[wrong].mean()}


rows = {}
for name, leaf_proba in oof_methods.items():
    rows[name] = evaluate(leaf_proba, node_id[-1], y_idx)
comparison = pd.DataFrame(rows).T.round(4)
comparison.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv")
print(comparison.to_string())
independent = oof_flat[-1].argmax(axis=1)
mismatch = np.zeros(len(train), dtype=bool)
for level in range(N_LEVELS - 1):
    mismatch |= oof_flat[level].argmax(axis=1) != ancestor[level][independent]
print("tebak tiap tingkat sendiri-sendiri → jalur tidak sah:", round(mismatch.mean(), 4))

Gambar banding cara (bahan slide). Pilihan `BEST` = skor daun OOF tertinggi.

In [ ]:
BEST = comparison["skor daun (metrik)"].idxmax()
# BEST = "gabung datar + rantai"          # alt: pilih sendiri (mis. lebih stabil)
shown = ["skor daun (metrik)", "F1 tingkat atas", "F1 hierarkis"]
positions = np.arange(len(comparison))
figure, ax = plt.subplots(figsize=(10, 4.5))
for number, column in enumerate(shown):
    bars = ax.bar(positions + (number - 1) * 0.27, comparison[column], width=0.27,
                  color=PALETTE[number], label=column)
    ax.bar_label(bars, fmt="%.3f", fontsize=7)
ax.set_xticks(positions)
ax.set_xticklabels(comparison.index, fontsize=9, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.12))
figure.savefig(f"{OUTPUT_FOLDER}/banding_metode.png", dpi=200, bbox_inches="tight")
plt.show()
print("dipilih:", BEST)

## Matriks bingung berurutan pohon
Daun diurutkan per induk, garis = batas induk. Kesalahan menumpuk di kotak diagonal berarti
salahnya "antar saudara". Pasangan tersering disimpan ke CSV.

In [ ]:
pred_leaf = oof_methods[BEST].argmax(axis=1)
pairs = pd.DataFrame({"asli": LEAVES[node_id[-1]], "tebakan": LEAVES[pred_leaf]})
pairs = pairs[node_id[-1] != pred_leaf]
top_pairs = pairs.value_counts().head(15)
top_pairs.rename("baris").to_csv(f"{OUTPUT_FOLDER}/salah_terbanyak.csv")
print(top_pairs.to_string())
if n_nodes[-1] <= MAX_PLOT_LEAVES:
    matrix_cm = confusion_matrix(node_id[-1], pred_leaf, labels=np.arange(n_nodes[-1]))
    matrix_cm = matrix_cm / np.clip(matrix_cm.sum(axis=1, keepdims=True), 1, None)
    figure, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(matrix_cm, cmap="Blues", vmin=0, vmax=1)
    for edge in np.flatnonzero(np.diff(ancestor[-2])) + 0.5:
        ax.axhline(edge, color="#D55E00", linewidth=0.8)
        ax.axvline(edge, color="#D55E00", linewidth=0.8)
    ax.set_xticks(np.arange(n_nodes[-1]))
    ax.set_xticklabels(leaf_short, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(n_nodes[-1]))
    ax.set_yticklabels(leaf_short, fontsize=7)
    ax.set_xlabel("tebakan")
    ax.set_ylabel("asli (baris dinormalkan)")
    figure.savefig(f"{OUTPUT_FOLDER}/bingung_hirarki.png", dpi=200, bbox_inches="tight")
    plt.show()

## Simpan OOF (kolom = kelas `TARGET`) + catat percobaan

In [ ]:
oof_class = (oof_methods[BEST] @ to_class).astype("float32")
test_class = (test_methods[BEST] @ to_class).astype("float32")
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof_class)
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_class)
SCORE = metric(y_idx, oof_class.argmax(axis=1))
log_path = f"{OOF_FOLDER}/percobaan.csv"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "klasifikasi_hierarkis", "model": MODEL_NAME,
                         "metrik": "f1_macro", "skor_cv": round(SCORE, 4), "detik": SECONDS,
                         "catatan": f"{BEST}, {' > '.join(LEVELS)}"}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("skor OOF:", round(SCORE, 4), "| disimpan:", f"oof_{MODEL_NAME}.npy", oof_class.shape)

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = train[TARGET].mode()[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):      # jangan timpa sample panitia
    demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Submission + cek

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(CLASSES[test_class.argmax(axis=1)], index=test[ID_COL].values)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
unknown = set(written[SUB_COL].astype(str)) - set(CLASSES)
if len(unknown) > 0:
    problems.append(f"label di luar train: {sorted(unknown)[:5]}")
spread = written[SUB_COL].value_counts()
print(spread.head(8).to_dict())
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)
Semua cara dinilai di test (daun di luar pohon train dihitung salah lewat kelas -1).

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].fillna("kosong").values, index=key[ID_COL].values)
truth = test[ID_COL].map(key_by_id)
truth = truth.astype(str).to_numpy()
truth_class = np.searchsorted(CLASSES, truth)
truth_class[~np.isin(truth, CLASSES)] = -1
rows = {}
for name, leaf_proba in test_methods.items():
    rows[name] = {"skor daun test": metric(truth_class, (leaf_proba @ to_class).argmax(axis=1))}
scores = pd.DataFrame(rows).T
print(scores.round(4).to_string(), "| OOF terpilih:", BEST, round(SCORE, 4))